In [ ]:
!pip install torch==1.7.1+cu110 torchvision==0.8.2+cu110 torchaudio==0.7.2 -f https://download.pytorch.org/whl/torch_stable.html

In [ ]:
import pandas as pd
import numpy as np
import torch
import time
import copy
import math
from datetime import datetime
from torch.utils.data import DataLoader

In [ ]:
data = pd.read_csv('/opt/neurothink/neuro-2e4bd43432a74c34a3d8261f430ef3b6/atlas-higgs-challenge-2014-v2.csv')
data['Binary_Label'] = data['Label'].map({'s':1,'b':0})

In [ ]:
class Feedforward(torch.nn.Module):
        def __init__(self, input_size, hidden_size, dropout):
            super(Feedforward, self).__init__()
            self.input_size = input_size
            self.hidden_size  = hidden_size
            self.fc_in = torch.nn.Linear(self.input_size, self.hidden_size)
            self.fc1 = torch.nn.Linear(self.hidden_size, self.hidden_size)
            self.fc2 = torch.nn.Linear(self.hidden_size, self.hidden_size)
            #self.relu = torch.nn.ReLU()
            self.relu = torch.nn.LeakyReLU(0.1)
            self.fc_out = torch.nn.Linear(self.hidden_size, 1)
            self.sigmoid = torch.nn.Sigmoid()
            self.dropout = torch.nn.Dropout(dropout)
            
        def forward(self, x):
            x = self.fc_in(x)
            x = self.dropout(x)
            x = self.relu(x)
            x = self.fc1(x)
            x = self.dropout(x)
            x = self.relu(x)
            x = self.fc2(x)
            x = self.dropout(x)
            x = self.relu(x)
            output = self.fc_out(x)
            output = self.sigmoid(output)
            return output

In [ ]:
nonphi_columns = [c for c in data.columns if "phi" in c]
for col in nonphi_columns:
    data.drop(col, axis=1, inplace=True)
    
# now replace all the '-999' values with NULL so we can properly handle them
#nulled_data = data.replace(-999.0, np.nan)
nulled_data = data.replace(-999.0, 0)
train_data = nulled_data.loc[nulled_data['KaggleSet'] == 't'].copy(deep=True)
public_test_data = nulled_data.loc[nulled_data['KaggleSet'] == 'b'].copy(deep=True)
private_test_data = nulled_data.loc[nulled_data['KaggleSet'] == 'v'].copy(deep=True)

orig_train_data = train_data.copy(deep=True)
orig_public_test_data = public_test_data.copy(deep=True)
orig_private_test_data = private_test_data.copy(deep=True)


# now replace the null values with column averages
train_data.drop('EventId', axis=1, inplace=True)
train_data.drop('Label', axis=1, inplace=True)
train_data.drop('Binary_Label', axis=1, inplace=True)
train_data.drop('Weight', axis=1, inplace=True)
train_data.drop('KaggleWeight', axis=1, inplace=True)
train_data.drop('KaggleSet', axis=1, inplace=True)
for col in train_data:
    train_data[col].fillna(train_data[col].mean(), inplace=True)
#train_data.isnull().sum()


public_test_data.drop('EventId', axis=1, inplace=True)
public_test_data.drop('Label', axis=1, inplace=True)
public_test_data.drop('Binary_Label', axis=1, inplace=True)
public_test_data.drop('Weight', axis=1, inplace=True)
public_test_data.drop('KaggleWeight', axis=1, inplace=True)
public_test_data.drop('KaggleSet', axis=1, inplace=True)
for col in public_test_data:
    public_test_data[col].fillna(public_test_data[col].mean(), inplace=True)
#public_test_data.isnull().sum()


private_test_data.drop('EventId', axis=1, inplace=True)
private_test_data.drop('Label', axis=1, inplace=True)
private_test_data.drop('Binary_Label', axis=1, inplace=True)
private_test_data.drop('Weight', axis=1, inplace=True)
private_test_data.drop('KaggleWeight', axis=1, inplace=True)
private_test_data.drop('KaggleSet', axis=1, inplace=True)
for col in private_test_data:
    private_test_data[col].fillna(private_test_data[col].mean(), inplace=True)
#private_test_data.isnull().sum()

# now let's normalize
normed_train_data = (train_data - train_data.min())/(train_data.max() - train_data.min())

normed_public_test_data = (public_test_data - public_test_data.min())/(public_test_data.max() - public_test_data.min())

normed_private_test_data = (private_test_data - private_test_data.min())/(private_test_data.max() - private_test_data.min())


train_input_data = []
for i in range(len(orig_train_data)):
   train_input_data.append([torch.tensor(normed_train_data.iloc[i], dtype=torch.float), torch.tensor(orig_train_data['Binary_Label'].iloc[i] , dtype=torch.float)])
valid_input_data = []
for i in range(len(orig_public_test_data)):
   valid_input_data.append([torch.tensor(normed_public_test_data.iloc[i], dtype=torch.float), torch.tensor(orig_public_test_data['Binary_Label'].iloc[i] , dtype=torch.float)])

from torch.utils.data import DataLoader

train_dataloader = DataLoader(train_input_data, batch_size=128, shuffle=True)
valid_dataloader = DataLoader(valid_input_data, batch_size=128, shuffle=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Feedforward(24, 600, 0.05)
model.to(device)
criterion = torch.nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr = 0.01)

import time
import copy
from datetime import datetime

now = datetime.now()
dt_string = now.strftime("%d%m%Y%H%M%S")
logfile = open(dt_string + ".log",'w')

start_time = time.time()
epoch = 40
maxvalcount = 5
minval_loss = np.inf
valcount = maxvalcount
best_model = model
for epoch in range(epoch):
    print("EPOCH: ", epoch)
    logfile.write("EPOCH: " + str(epoch) + "\n")
    model.train()
    for i,batch in enumerate(train_dataloader):
        inputs, output = batch
        inputs, output = inputs.to(device), output.to(device)
        optimizer.zero_grad()
        # Forward pass
        y_pred = model(inputs)
        # Compute Loss
        loss = criterion(y_pred.squeeze(), output)
        if i % 100 == 0:
            logstring = 'Batch {}: train loss: {}'.format(i, loss.item())
            print(logstring)
            logfile.write(logstring + '\n')
        # Backward pass
        loss.backward()
        optimizer.step()
    # compute validation loss
    model.eval()
    with torch.set_grad_enabled(False):
        val_loss = 0
        for i,batch in enumerate(valid_dataloader):
            inputs, output = batch
            inputs, output = inputs.to(device), output.to(device)
            y_pred = model(inputs)
            y_pred = model(inputs)
            loss = criterion(y_pred.squeeze(), output)
            val_loss += loss
            if i % 100 == 0:
                logstring = 'Validation Batch {}: loss: {}'.format(i, loss.item())
                print(logstring)
                logfile.write(logstring + '\n')
        avg_val_loss = val_loss / len(valid_dataloader)
        valstring = f"AVERAGE BATCH VAL LOSS = {avg_val_loss}"
        print(valstring)
        logfile.write(valstring + '\n')
    if avg_val_loss < minval_loss:
        minval_loss = avg_val_loss
        best_model = copy.deepcopy(model)
        valcount = maxvalcount
    else:
        valcount -= 1
    if valcount == 0:
        endmsg = f"Validation Loss failed to decrease in {maxvalcount} epochs, exiting with best model, val loss = {minval_loss}"
        print(endmsg)
        logfile.write(endmsg + '\n')
        break

end_time = time.time()
timestring = f"Time elapsed in training: {end_time - start_time} seconds"
print(timestring)
logfile.write(timestring + '\n')

torch.save(best_model, 'mlp.600.nophi.pt')

In [ ]:
def AMS(s, b):
    """ Approximate Median Significance defined as:
        AMS = sqrt(
                2 { (s + b + b_r) log[1 + (s/(b+b_r))] - s}
              )        
    where b_r = 10, b = background, s = signal, log is natural logarithm """
    
    br = 10.0
    radicand = 2 *( (s+b+br) * math.log (1.0 + s/(b+br)) -s)
    if radicand < 0:
        print('radicand is negative. Exiting')
        exit()
    else:
        return math.sqrt(radicand)
    
def compute_performance(logfile, setname, y_pred, y_gold, y_weight):
    s = 0
    b = 0
    tp = 0
    fp = 0
    for i,y in enumerate(y_pred):
        if y[0] > 0.5:
            if y_gold.iloc[i] == 's':
                s += y_weight.iloc[i]
                tp += 1
            else:
                b += y_weight.iloc[i]
                fp += 1

    out1 = f"{setname}: (S,B) = ({s:.3f},{b:.3f}), (TP,FP) = {tp}, {fp}"
    print(out1)
    logfile.write(out1 + '\n')

    outstring = f"{setname}: (S,B) = ({s:.3f},{b:.3f}), AMS={AMS(s,b):.3f}, Unweighted Precision = {tp/(tp+fp):.3f}"
    print(outstring)
    logfile.write(outstring + '\n')

In [ ]:
y_pred = best_model(torch.tensor(normed_train_data.values, dtype=torch.float).to(device)).tolist()
y_gold = orig_train_data['Label']
y_weight = orig_train_data['KaggleWeight']
compute_performance(logfile, "TRAINING", y_pred, y_gold, y_weight)

y_pred = best_model(torch.tensor(normed_public_test_data.values, dtype=torch.float).to(device)).tolist()
y_gold = orig_public_test_data['Label']
y_weight = orig_public_test_data['KaggleWeight']
compute_performance(logfile, "VALIDATION", y_pred, y_gold, y_weight)

# Test set performance
y_pred = best_model(torch.tensor(normed_private_test_data.values, dtype=torch.float).to(device)).tolist()
y_gold = orig_private_test_data['Label']
y_weight = orig_private_test_data['KaggleWeight']
compute_performance(logfile, "TESTING", y_pred, y_gold, y_weight)

In [ ]:
logfile.close()